# Classical Control and Motion Planning

## Table of Contents

**Theory & demos:**
1. [Levels of Control](#levels-of-control)
2. [Closed-Loop Control](#closed-loop-control)
3. [Linear Systems](#linear-systems)
4. [Linearisation](#linearisation)
5. [CartPole Environment (practice setup)](#cartpole-environment)
6. [PID Control](#pid-control)
7. [LQR](#lqr)
8. [MPC](#mpc)
9. [Lyapunov & Energy-Based Control](#lyapunov)
10. [Reinforcement Learning](#rl)
11. [Summary & Comparisons](#summary)

**Practice tasks** are interleaved: after each method's theory you will run and tune controllers on the same CartPole environment.


In [ ]:
%matplotlib widget
import sys
from pathlib import Path

_week_dir = Path.cwd()
if str(_week_dir) not in sys.path:
    sys.path.insert(0, str(_week_dir))

import numpy as np
import matplotlib.pyplot as plt

---
# 1. Levels of Control in Modern Robots <a id="levels-of-control"></a>

Before we dive into individual control methods, let's zoom out and look at **how real robots are controlled**.

Every robot — from a factory arm to a self-driving car — runs a **hierarchy of controllers**, each operating at a different level of abstraction and a different time-scale:

| Layer | What it decides | Typical rate | Example methods |
|-------|----------------|-------------|----------------|
| **Task / Mission** | *What* to do ("pick up cup", "go to room B") | seconds | FSM, Behavior Trees, LLM planners |
| **Motion Planning** | *How* to get there (collision-free path) | 1–10 Hz | RRT, A*, trajectory optimisation |
| **Trajectory Tracking / MPC** | Follow the plan under dynamics | 10–100 Hz | MPC, LQR, feedback linearisation |
| **Low-level Servo** | Individual joint/motor control | 1–10 kHz | PID, current control |

## Real-World Examples <a id="real-world-examples"></a>

| System | Task layer | Planning | Tracking | Servo |
|--------|-----------|----------|----------|-------|
| Industrial arm | Task planner | Trajectory generator | Joint-space tracking | Joint PID |
| Quadcopter | Mission planner | Path planner | Attitude MPC | Motor ESC PID |
| Self-driving car | Route planner | Behavior planner | MPC | Steering/throttle PID |
| Humanoid | Gait planner | Whole-body MPC | Torque control | Motor current loops |

A humanoid robot is essentially an **inverted pendulum** — and many real-world systems reduce to the same abstraction:

<div style="display: flex; gap: 20px; align-items: center; justify-content: center;">
    <figure style="text-align: center;">
        <img src="assets/humanoid_is_inverted_pendulum.png" width="300">
        <figcaption><em>Humanoid ≈ Inverted pendulum</em></figcaption>
    </figure>
    <figure style="text-align: center;">
        <img src="assets/mono_wheel_is_cart_pole.jpeg" width="300">
        <figcaption><em>Monowheel ≈ Cart-pole</em></figcaption>
    </figure>
</div>

The **cart-pole** will be our running example throughout this lecture.

<video width="600" controls autoplay loop>
    <source src="assets/my_personal_life.mp4" type="video/mp4">
</video>

## Bellman's Principle of Optimality <a id="bellman-principle"></a>

> *"An optimal policy has the property that whatever the initial state and initial decision are, the remaining decisions must constitute an optimal policy with regard to the state resulting from the first decision."*  
> — Richard Bellman, 1957

This deceptively simple idea is the **unifying thread** of this lecture:

- It lets us decompose optimal control into **stages** (dynamic programming).
- In **continuous time** with a known model → Hamilton-Jacobi-Bellman (HJB) equation → **LQR** (Section 6).
- In **discrete time** with an unknown model → Bellman equation → **Reinforcement Learning** (Section 9).
- **MPC** solves a truncated Bellman problem online at each step (Section 7).
- The control hierarchy itself can be viewed through this lens: each layer solves a sub-problem *optimally given what the layer above decided*.

| Method | Layer | Model-based | Typical Rate |
|--------|-------|-------------|--------------|
| PID | Low-level Servo | Yes | 1–10 kHz |
| LQR | Trajectory Tracking | Yes | 10–100 Hz |
| MPC | Trajectory Tracking | Yes | 10–100 Hz |
| Lyapunov | Trajectory Tracking | Yes | 10–100 Hz |
| IK | Motion Planning | Yes | 1–10 Hz |
| RL | Task / Motion Planning | No | 1–100 Hz |

In [ ]:
from lib.control_hierarchy import show_control_hierarchy
show_control_hierarchy()


---
# 2. The Closed-Loop Control Problem <a id="closed-loop-control"></a>

## Open-Loop vs Closed-Loop <a id="open-vs-closed-loop"></a>

**Open-loop control** applies a pre-computed input $u(t)$ without measuring the actual state:

$$\text{Reference } r(t) \;\longrightarrow\; \boxed{\text{Controller}} \;\longrightarrow\; u(t) \;\longrightarrow\; \boxed{\text{System}} \;\longrightarrow\; y(t)$$

**Closed-loop (feedback) control** measures the output and computes the error $e(t) = r(t) - y(t)$:

$$\boxed{r(t)  - e(t)} \;\longrightarrow\; \boxed{\text{Controller}} \;\longrightarrow\; u(t) \;\longrightarrow\; \boxed{\text{System}} \;\longrightarrow\; y(t) \;\longrightarrow\; \text{(feedback)}$$

Why does feedback matter? Consider an **unstable system** $\dot{x} = ax$ with $a > 0$:
- Open-loop: any perturbation grows exponentially
- Closed-loop with $u = kx$: effective dynamics become $\dot{x} = (a + bk)x$, and choosing $k$ such that $a + bk < 0$ **stabilises** the system

Below: state $x(t)$ for open-loop vs closed-loop ($\dot{x} = ax + bu$, $a=1$, $b=1$). The closed-loop plot also shows the control input $u(t) = kx$.

In [ ]:
from lib.linear_systems import show_open_vs_closed_loop
show_open_vs_closed_loop()

## Why Feedback Is Essential <a id="why-feedback"></a>

In practice, open-loop fails because of:

- **Disturbances**: wind, friction, unexpected loads
- **Model uncertainty**: the real system never matches the model exactly
- **Sensor noise**: measurements are corrupted
- **Delays**: actuation takes time; digital controllers run at finite rates
- **Saturation**: actuators have physical limits ($|u| \le u_{\max}$)

Feedback provides **disturbance rejection** and **robustness** to model errors.

## Analytic Example: First-Order System <a id="first-order-example"></a>

System: $\dot{x} = -a x + b u$, with $a = 1, b = 1$ (unstable open-loop pole at $s = +1$).

Proportional feedback: $u = K_p (r - x)$.

Closed-loop: $\dot{x} = -(a + b K_p)x + b K_p r$.

Transfer function: $\frac{X(s)}{R(s)} = \frac{b K_p}{s + a + b K_p}$

The closed-loop pole is at $s = -(a + bK_p)$. For $K_p > -a/b = -1$, the system is stable. Larger $K_p$ makes the pole more negative → faster response, but at the cost of larger control effort.

## CartPole: Open-Loop Fails <a id="cartpole-open-loop"></a>

In [ ]:
from lib.cartpole_sim import simulate_cartpole, animate_cartpole, DEFAULT_PARAMS

state0 = np.array([0.0, 0.1, 0.0, 0.0])  # small perturbation
ts, states, controls = simulate_cartpole(state0, lambda t, s: 0.0, T=3.0)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ts, np.degrees(states[:, 1]), 'r', lw=2)
ax.set_xlabel('t [s]'); ax.set_ylabel('θ [deg]')
ax.set_title('Open-loop CartPole: θ₀ = 0.1 rad — pole falls immediately')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
animate_cartpole(ts, states, controls)

> **Quick Check:** Why can't we simply apply a constant force to keep the cart-pole upright?

---
# 3. Linear Systems <a id="linear-systems"></a>

## State-Space Representation <a id="state-space"></a>

A **linear time-invariant (LTI)** system is described by:

$$\dot{x} = Ax + Bu$$

where $x \in \mathbb{R}^n$ is the **state**, $u \in \mathbb{R}^m$ is the **input**, and $y \in \mathbb{R}^p$ is the **output**.

This representation is fundamental because:
1. It handles **MIMO** (multi-input, multi-output) systems naturally
2. It connects directly to **matrix theory** (eigenvalues, stability, controllability)
3. **Nonlinear** systems can be **linearised** into this form (Section 4)

## Stability <a id="stability"></a>

The system $\dot{x} = Ax$ (no input) has solution $x(t) = e^{At} x_0$.

**Stability depends entirely on the eigenvalues of $A$:**

| All eigenvalues have... | System is... |
|-------------------------|-------------|
| $\text{Re}(\lambda_i) < 0$ | **Asymptotically stable** (trajectories → 0) |
| $\text{Re}(\lambda_i) \le 0$ (some $= 0$) | **Marginally stable** (bounded but not converging) |
| Any $\text{Re}(\lambda_i) > 0$ | **Unstable** (trajectories diverge) |

<details>
<summary><b>Derivation: Matrix Exponential and Stability (click to expand)</b></summary>

**Defining $e^{At}$ via Taylor series.** For a square matrix $A \in \mathbb{R}^{n \times n}$, define the matrix exponential by the power series

$$e^{At} \;=\; I + At + \frac{A^2 t^2}{2!} + \frac{A^3 t^3}{3!} + \cdots \;=\; \sum_{k=0}^{\infty} \frac{(At)^k}{k!}.$$

This series converges absolutely for every $A$ and every $t \in \mathbb{R}$ (by the ratio test on $\|A^k t^k / k!\|$).

**$e^{At}$ solves $\dot{x} = Ax$.** Differentiate term-by-term:

$$\frac{d}{dt} e^{At} = \frac{d}{dt}\sum_{k=0}^{\infty} \frac{A^k t^k}{k!} = \sum_{k=1}^{\infty} \frac{A^k t^{k-1}}{(k-1)!} = A \sum_{j=0}^{\infty} \frac{A^j t^j}{j!} = A\, e^{At}.$$

So $x(t) = e^{At} x_0$ satisfies $\dot{x}(t) = A\, e^{At} x_0 = A\, x(t)$ with $x(0) = e^{A \cdot 0} x_0 = I\, x_0 = x_0$. $\square$

**Diagonalizable case.** Suppose $A$ has $n$ linearly independent eigenvectors, so $A = P D P^{-1}$ where $D = \text{diag}(\lambda_1, \ldots, \lambda_n)$ and $P$ is the matrix of eigenvectors. Then $A^k = P D^k P^{-1}$, and

$$e^{At} = \sum_{k=0}^{\infty} \frac{P D^k P^{-1}\, t^k}{k!} = P \left(\sum_{k=0}^{\infty} \frac{D^k t^k}{k!}\right) P^{-1} = P\, e^{Dt}\, P^{-1}$$

where $e^{Dt} = \text{diag}(e^{\lambda_1 t}, \ldots, e^{\lambda_n t})$, since the matrix exponential of a diagonal matrix is simply the exponential applied entry-wise. So the solution decomposes as

$$x(t) = P\, e^{Dt}\, P^{-1} x_0 = \sum_{i=1}^{n} c_i\, e^{\lambda_i t}\, v_i$$

where $v_i$ are the eigenvectors of $A$ and $c_i = (P^{-1} x_0)_i$ are the coefficients of $x_0$ in the eigenbasis. Each **mode** $c_i e^{\lambda_i t} v_i$ evolves independently.

**Stability from eigenvalues.** Write the eigenvalues as $\lambda_i = \sigma_i + j\omega_i$ where $\sigma_i = \text{Re}(\lambda_i)$ and $\omega_i = \text{Im}(\lambda_i)$. Then

$$e^{\lambda_i t} = e^{\sigma_i t}\left(\cos(\omega_i t) + j \sin(\omega_i t)\right)$$

so $|e^{\lambda_i t}| = e^{\sigma_i t}$. This gives:
- $\sigma_i < 0$: mode $i$ decays exponentially $\to 0$ as $t \to \infty$
- $\sigma_i = 0$: mode $i$ oscillates with constant amplitude (bounded, not decaying)
- $\sigma_i > 0$: mode $i$ grows exponentially $\to \infty$

Since $x(t) = \sum_i c_i e^{\lambda_i t} v_i$, the state converges to zero for all initial conditions iff **every** mode decays, i.e. $\text{Re}(\lambda_i) < 0$ for all $i$. A single positive real part makes the system unstable. $\square$

**Non-diagonalizable case (Jordan form).** When $A$ is not diagonalizable, the Jordan decomposition $A = P J P^{-1}$ introduces Jordan blocks with polynomial factors:

$$e^{Jt} = \text{blockdiag}\!\left(e^{\lambda_i t}\begin{bmatrix} 1 & t & t^2/2 & \cdots \\ 0 & 1 & t & \cdots \\ \vdots & & \ddots & \end{bmatrix}\right)$$

The polynomial factors $t^k$ grow, but are always dominated by $e^{\sigma_i t}$ when $\sigma_i < 0$ (since $\lim_{t \to \infty} t^k e^{\sigma t} = 0$ for $\sigma < 0$). So the stability criterion $\text{Re}(\lambda_i) < 0$ for all $i$ remains correct even for non-diagonalizable matrices. For $\text{Re}(\lambda_i) = 0$ with a non-trivial Jordan block, the polynomial growth makes the system **unstable** (not just marginally stable).

</details>

In [ ]:
from lib.linear_systems import show_stability_examples
show_stability_examples()

## Interactive: How Eigenvalues Shape System Response <a id="eigenvalue-interactive"></a>

In [ ]:
from lib.linear_systems import show_eigenvalue_response_interactive
show_eigenvalue_response_interactive()

## Controllability <a id="controllability"></a>

Can we steer the system from **any** initial state to **any** target state using the input $u$?

**Controllability matrix:**

$$\mathcal{C} = \begin{bmatrix} B & AB & A^2B & \cdots & A^{n-1}B \end{bmatrix}$$

The system $(A, B)$ is **controllable** if and only if $\text{rank}(\mathcal{C}) = n$.

Intuition: each column $A^k B$ represents the directions the input can influence after $k$ steps through the dynamics.

<details>
<summary><b>Derivation: Why Only $n$ Columns Suffice (Cayley-Hamilton) (click to expand)</b></summary>

We claimed that the controllability matrix $\mathcal{C} = [B \;\; AB \;\; A^2 B \;\; \cdots \;\; A^{n-1}B]$ captures all directions the input can ever reach. Why do we stop at $A^{n-1}B$ and not continue to $A^n B, A^{n+1}B, \ldots$?

**Cayley-Hamilton theorem.** Every square matrix $A \in \mathbb{R}^{n \times n}$ satisfies its own characteristic polynomial: if $p(\lambda) = \det(\lambda I - A) = \lambda^n + c_{n-1}\lambda^{n-1} + \cdots + c_1 \lambda + c_0$, then

$$p(A) = A^n + c_{n-1} A^{n-1} + \cdots + c_1 A + c_0 I = 0.$$

This means $A^n = -c_{n-1} A^{n-1} - \cdots - c_1 A - c_0 I$, so $A^n$ is a linear combination of $I, A, \ldots, A^{n-1}$. By induction, **every** power $A^k$ with $k \ge n$ is a linear combination of $I, A, \ldots, A^{n-1}$. Therefore $A^k B$ for $k \ge n$ lies in $\text{colspan}(B, AB, \ldots, A^{n-1}B)$.

**Reachable set.** The solution of $\dot{x} = Ax + Bu$ with $x(0) = 0$ at time $t$ is

$$x(t) = \int_0^t e^{A(t-\tau)} B\, u(\tau)\, d\tau.$$

Expanding the matrix exponential:

$$e^{A(t-\tau)} B = \sum_{k=0}^{\infty} \frac{(t-\tau)^k}{k!} A^k B.$$

By Cayley-Hamilton, every $A^k B$ with $k \ge n$ is a linear combination of $B, AB, \ldots, A^{n-1}B$. So $e^{A(t-\tau)} B$ lies in $\text{colspan}(B, AB, \ldots, A^{n-1}B) = \text{Im}(\mathcal{C})$ for every $\tau$. The integral (a continuous sum of vectors in $\text{Im}(\mathcal{C})$) therefore also lies in $\text{Im}(\mathcal{C})$.

**Conclusion.** The set of states reachable from the origin is exactly $\text{Im}(\mathcal{C})$. The system is controllable (can reach any state) iff $\text{Im}(\mathcal{C}) = \mathbb{R}^n$, i.e. $\text{rank}(\mathcal{C}) = n$. $\square$

**Remark.** For the converse direction (every direction in $\text{Im}(\mathcal{C})$ is actually reachable), one constructs an explicit input $u(\tau)$ using $u(\tau) = B^\top e^{A^\top(t-\tau)} W_t^{-1} x_f$ where $W_t = \int_0^t e^{A\tau} B B^\top e^{A^\top \tau} d\tau$ is the **controllability Gramian**, which is invertible iff $\text{rank}(\mathcal{C}) = n$.

</details>

In [ ]:
from lib.linear_systems import show_controllability_demo
show_controllability_demo()

> **Quick Check:** A system has $A = \text{diag}(1, 2)$ and $B = [1, 0]^\top$. Is it controllable? What does this mean physically?

---
# 4. Linearisation <a id="linearisation"></a>

Most real systems are **nonlinear**: $\dot{x} = f(x, u)$.

We can approximate this as a linear system near an **equilibrium point** $(x^*, u^*)$ where $f(x^*, u^*) = 0$:

$$\dot{\delta x} \approx A\, \delta x + B\, \delta u$$

where the **Jacobians** are:

$$A = \frac{\partial f}{\partial x}\bigg|_{x^*, u^*}, \qquad B = \frac{\partial f}{\partial u}\bigg|_{x^*, u^*}$$

## CartPole Linearisation <a id="cartpole-linearisation"></a>

The cart-pole has state $x = [x, \theta, \dot{x}, \dot{\theta}]^\top$ and control $u$ (force on the cart).

We linearise around the **upright equilibrium** $\theta = 0, \dot\theta = 0, \dot{x} = 0$, where $\sin\theta \approx \theta$ and $\cos\theta \approx 1$.

<details>
<summary><b>Derivation: Linearisation via Multivariable Taylor Expansion (click to expand)</b></summary>

**Setup.** Consider a nonlinear system $\dot{x} = f(x, u)$ with an equilibrium at $(x^*, u^*)$, meaning $f(x^*, u^*) = 0$. We want to approximate the dynamics for small deviations $\delta x = x - x^*$, $\delta u = u - u^*$.

**Multivariable Taylor expansion.** For $f: \mathbb{R}^n \times \mathbb{R}^m \to \mathbb{R}^n$ that is twice continuously differentiable, the first-order Taylor expansion about $(x^*, u^*)$ gives:

$$f(x^* + \delta x,\; u^* + \delta u) = f(x^*, u^*) + \frac{\partial f}{\partial x}\bigg|_{(x^*, u^*)} \delta x + \frac{\partial f}{\partial u}\bigg|_{(x^*, u^*)} \delta u + \mathcal{R}(\delta x, \delta u)$$

where the **Jacobians** are

$$\frac{\partial f}{\partial x}\bigg|_{(x^*, u^*)} \in \mathbb{R}^{n \times n}, \qquad \left(\frac{\partial f}{\partial x}\right)_{ij} = \frac{\partial f_i}{\partial x_j}\bigg|_{(x^*, u^*)}$$

and similarly for $\frac{\partial f}{\partial u} \in \mathbb{R}^{n \times m}$.

**Remainder bound.** The remainder $\mathcal{R}$ satisfies

$$\|\mathcal{R}(\delta x, \delta u)\| \le \frac{M}{2}\left(\|\delta x\| + \|\delta u\|\right)^2$$

where $M = \sup \|D^2 f\|$ is a bound on the second derivatives of $f$ in a neighbourhood of $(x^*, u^*)$. This is the standard multivariate Taylor remainder theorem.

**Linearised dynamics.** Since $f(x^*, u^*) = 0$ at equilibrium, and $\dot{x} = \dot{\delta x}$ (because $x^*$ is constant):

$$\dot{\delta x} = \underbrace{\frac{\partial f}{\partial x}\bigg|_{*}}_{A}\, \delta x + \underbrace{\frac{\partial f}{\partial u}\bigg|_{*}}_{B}\, \delta u + \mathcal{R}(\delta x, \delta u)$$

Dropping the remainder gives the linear approximation $\dot{\delta x} \approx A\, \delta x + B\, \delta u$.

**Validity.** The approximation error is $O(\|\delta x\|^2 + \|\delta u\|^2)$. Concretely, if the state deviation has magnitude $\epsilon$, the neglected terms scale as $\epsilon^2$. When $\epsilon$ is small, $\epsilon^2 \ll \epsilon$, so the linear term dominates. But when $\epsilon$ grows (e.g. the 50° experiment below), the quadratic and higher-order terms become significant and the linearisation breaks down. The "radius of validity" depends on the curvature of $f$ — highly nonlinear systems (sharp $\sin\theta$ bends, etc.) have a smaller valid region.

**Hartman-Grobman theorem.** If all eigenvalues of $A$ have nonzero real part (the equilibrium is **hyperbolic**), then the nonlinear system $\dot{x} = f(x)$ is **topologically equivalent** to the linear system $\dot{\delta x} = A\, \delta x$ near the equilibrium. This means the qualitative behavior (stable/unstable manifolds, phase portrait topology) of the linearised system matches the nonlinear system locally — not just the trajectories, but the entire phase portrait structure.

</details>

In [ ]:
from lib.cartpole_sim import linearize_cartpole, CartPoleParams, DEFAULT_PARAMS

A, B = linearize_cartpole()
print("A =")
print(A)
print("\nB =")
print(B)
print(f"\nOpen-loop eigenvalues: {np.linalg.eigvals(A)}")
print("→ One positive eigenvalue → the upright equilibrium is UNSTABLE")

## When Does Linearisation Break Down? <a id="linearisation-breakdown"></a>

The linear model is only valid **near** the equilibrium. Let's compare nonlinear vs linearised responses for different initial angles:

In [ ]:
from lib.cartpole_sim import simulate_cartpole, solve_lqr

A, B = linearize_cartpole()
Q = np.diag([1, 10, 1, 10])
R = np.array([[1.0]])
K, _ = solve_lqr(A, B, Q, R)
lqr_ctrl = lambda t, s: float(-K @ s)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, theta0_deg in zip(axes, [5, 20, 50]):
    theta0 = np.radians(theta0_deg)
    state0 = np.array([0.0, theta0, 0.0, 0.0])

    # nonlinear
    ts_nl, st_nl, _ = simulate_cartpole(state0, lqr_ctrl, T=5.0)

    # linearised simulation
    A_cl = A - B @ K
    dt = 0.02
    n = int(5.0 / dt)
    st_lin = np.zeros((n+1, 4)); st_lin[0] = state0
    for i in range(n):
        st_lin[i+1] = st_lin[i] + dt * (A_cl @ st_lin[i])

    ax.plot(ts_nl, np.degrees(st_nl[:, 1]), 'b', lw=2, label='Nonlinear')
    ax.plot(ts_nl, np.degrees(st_lin[:, 1]), 'r--', lw=2, label='Linearised')
    ax.set_title(f'θ₀ = {theta0_deg}°')
    ax.set_xlabel('t [s]'); ax.set_ylabel('θ [deg]')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Nonlinear vs Linearised CartPole (with LQR controller)', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
theta0_anim = np.radians(20)
state0_anim = np.array([0.0, theta0_anim, 0.0, 0.0])
ts_anim, states_anim, controls_anim = simulate_cartpole(state0_anim, lqr_ctrl, T=5.0)
animate_cartpole(ts_anim, states_anim, controls_anim)

<details>
<summary><b>Full CartPole EOM Derivation (click to expand)</b></summary>

Using the **Euler-Lagrange** approach with generalised coordinates $q = (x, \theta)$:

**Kinetic energy:**

$$T = \frac{1}{2} m_c \dot{x}^2 + \frac{1}{2} m_p \left[(\dot{x} + l\dot{\theta}\cos\theta)^2 + (l\dot{\theta}\sin\theta)^2\right]$$

**Potential energy:**

$$V = m_p g l \cos\theta$$

**Lagrangian:** $\mathcal{L} = T - V$

Applying $\frac{d}{dt}\frac{\partial \mathcal{L}}{\partial \dot{q}_i} - \frac{\partial \mathcal{L}}{\partial q_i} = Q_i$ (generalised forces) and solving for the accelerations:

$$\ddot{\theta} = \frac{(m_c + m_p) g \sin\theta - \cos\theta \left(u + m_p l \dot{\theta}^2 \sin\theta\right)}{l\left(m_c + m_p - m_p\cos^2\theta\right)}$$

$$\ddot{x} = \frac{u + m_p l\left(\dot{\theta}^2 \sin\theta - \ddot{\theta}\cos\theta\right)}{m_c + m_p}$$

Linearising around $\theta = 0$ ($\sin\theta \approx \theta$, $\cos\theta \approx 1$, $\dot{\theta}^2 \approx 0$):

$$A = \begin{bmatrix} 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & \frac{m_p g}{m_c} & 0 & 0 \\ 0 & \frac{(m_c+m_p)g}{m_c l} & 0 & 0 \end{bmatrix}, \quad B = \begin{bmatrix} 0 \\ 0 \\ \frac{1}{m_c} \\ \frac{1}{m_c l} \end{bmatrix}$$

</details>

> **Quick Check:** The linearised CartPole A matrix has two zero eigenvalues and one positive eigenvalue. What does each one correspond to physically?

---

## CartPole Environment <a id="cartpole-environment"></a>

The `CartPoleEnv` class encapsulates the full nonlinear dynamics, RK4 integration, reward computation, and rendering.

**State:** $x = [x, \theta, \dot{x}, \dot{\theta}]^\top$ where $\theta = 0$ is the **upright** position.

**Control:** $u \in [-f_{\max}, f_{\max}]$ — horizontal force on the cart.

Let's start by creating an environment and exploring its behaviour with zero control.

In [ ]:
from lib.cartpole_env import CartPoleEnv, animate_episode
from lib.controllers import ZeroController
from lib.plotting import plot_episode

TARGET_X = 1.0  # configurable: target cart position (pole upright)

env = CartPoleEnv(target_x=TARGET_X)
print(f"Parameters: m_cart={env.m_cart}, m_pole={env.m_pole}, l={env.l}, g={env.g}")
print(f"Simulation step: dt={env.dt}")
print(f"Force limit: ±{env.f_max} N")
print(f"Target pose: x={env.target_x}, θ=0 (upright)")

In [ ]:
zero_ctrl = ZeroController()
initial_state = np.array([0.0, 0.1, 0.0, 0.0])  # small angle perturbation
ts, states, controls, rewards = env.run_episode(zero_ctrl, T=3.0, initial_state=initial_state)
plot_episode(ts, states, controls, rewards, title="Zero controller: pole falls", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts, states, controls, env)

### Verify the dynamics <a id="verify-dynamics"></a>

Let's check that the linearisation gives us the expected A and B matrices:

In [ ]:
A, B = env.linearize()
print("A =")
print(A)
print("\nB =")
print(B)
print(f"\nOpen-loop eigenvalues: {np.linalg.eigvals(A)}")

One positive eigenvalue confirms the upright equilibrium is **unstable** — we need feedback control.

Also check the energy at the upright position — we'll need this for swing-up:

In [ ]:
print(f"Energy at upright (theta=0): {env.upright_energy():.4f}")
print(f"Energy at bottom (theta=pi): {env.energy(np.array([0, np.pi, 0, 0])):.4f}")

---
# 5. PID Control <a id="pid-control"></a>

The **PID controller** is the workhorse of industrial control. Over 95% of all control loops in industry use some form of PID.

## From P to PID <a id="p-to-pid"></a>

Given error $e(t) = r(t) - x(t)$:

**P-controller:** $u(t) = K_p \, e(t)$
- Fast response but often has **steady-state error** (if the system has no integrator)

**PI-controller:** $u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\,d\tau$
- The integral term **eliminates steady-state error** by accumulating past errors
- But can cause **overshoot** and **windup** if the integral grows too large

**PID-controller:** $u(t) = K_p \, e(t) + K_i \int_0^t e(\tau)\,d\tau + K_d \, \dot{e}(t)$
- The derivative term provides **damping** — it anticipates future error by reacting to the rate of change
- Reduces overshoot but is sensitive to noise

## Discrete-Time PID <a id="discrete-pid"></a>

In practice, controllers run on digital computers at sampling period $T_s$:

$$u[k] = K_p \, e[k] \;+\; K_i \, T_s \sum_{j=0}^{k} e[j] \;+\; K_d \, \frac{e[k] - e[k-1]}{T_s}$$

**Practical considerations:**
- **Derivative filtering:** raw $\dot{e}$ amplifies noise → use a low-pass filtered derivative: $d_f[k] = \alpha \, d_f[k-1] + (1-\alpha)\frac{e[k]-e[k-1]}{T_s}$
- **Integral anti-windup:** clamp the integral term to prevent unbounded growth when the actuator saturates
- **Derivative kick:** when the setpoint changes, $\dot{e}$ spikes → differentiate the *measurement* instead of the error

Below: mass-spring-damper step response. Tune Kp, Ki, Kd with the sliders to see the effect on rise time, overshoot, and settling.

<details>
<summary><b>Derivation: PID in the Laplace Domain (click to expand)</b></summary>

### Laplace Transform and Transfer Functions

The **Laplace transform** of a signal $f(t)$ (defined for $t \ge 0$) is $F(s) = \int_0^\infty f(t)\, e^{-st}\, dt$ where $s \in \mathbb{C}$. Key properties:

$$\mathcal{L}\{\dot{f}\} = s F(s) - f(0), \qquad \mathcal{L}\left\{\int_0^t f(\tau)\, d\tau\right\} = \frac{F(s)}{s}$$

**PID transfer function.** Taking the Laplace transform of $u(t) = K_p e(t) + K_i \int_0^t e(\tau)\, d\tau + K_d \dot{e}(t)$ with zero initial conditions:

$$C(s) = \frac{U(s)}{E(s)} = K_p + \frac{K_i}{s} + K_d s = \frac{K_d s^2 + K_p s + K_i}{s}$$

### Closed-Loop Transfer Function

For a plant with transfer function $G(s)$ in a unity feedback loop with controller $C(s)$:

$$\frac{Y(s)}{R(s)} = \frac{C(s)\, G(s)}{1 + C(s)\, G(s)}, \qquad E(s) = R(s) - Y(s) = \frac{R(s)}{1 + C(s)\, G(s)}$$

The denominator $1 + C(s)G(s)$ is the **characteristic polynomial** — its roots (the closed-loop poles) determine stability and transient response.

### Steady-State Error via the Final Value Theorem

**Final Value Theorem:** If all poles of $sF(s)$ have negative real parts, then $\lim_{t \to \infty} f(t) = \lim_{s \to 0} s\, F(s)$.

For a unit step reference $R(s) = 1/s$, the steady-state error is:

$$e_{ss} = \lim_{s \to 0} s \cdot E(s) = \lim_{s \to 0} s \cdot \frac{1/s}{1 + C(s)G(s)} = \lim_{s \to 0} \frac{1}{1 + C(s)G(s)}$$

**P-only control** ($C(s) = K_p$) on a **type-0 plant** (no free integrators, $G(0) = K_{\text{dc}} < \infty$):

$$e_{ss} = \frac{1}{1 + K_p\, K_{\text{dc}}} \neq 0$$

There is always a nonzero steady-state error. Increasing $K_p$ reduces it but never eliminates it.

**PI control** ($C(s) = K_p + K_i/s$): As $s \to 0$, $C(s) \to \infty$ because of the $K_i/s$ term. So:

$$e_{ss} = \lim_{s \to 0} \frac{1}{1 + C(s)G(s)} = \frac{1}{1 + \infty} = 0$$

The integral action introduces a pole at $s = 0$ in the controller, making the open-loop transfer function $C(s)G(s)$ a **type-1** system. By the internal model principle, a type-1 loop tracks a step input with zero steady-state error. $\square$

### Characteristic Polynomial and Pole Placement

Consider a second-order plant $G(s) = \frac{1}{ms^2 + cs + k}$ (mass-spring-damper). With PID control $C(s) = K_p + K_i/s + K_d s$, the closed-loop characteristic polynomial is:

$$\Delta(s) = ms^3 + (c + K_d)s^2 + (k + K_p)s + K_i = 0$$

This is a **cubic** in $s$. By choosing $K_p, K_i, K_d$ we can place the three closed-loop poles independently (up to the constraint that complex poles come in conjugate pairs):

- **$K_d$** adds to the $s^2$ coefficient → controls **damping** (analogous to increasing the damping ratio $\zeta$)
- **$K_p$** adds to the $s^1$ coefficient → controls **stiffness / natural frequency** $\omega_n$
- **$K_i$** sets the $s^0$ coefficient → determines **low-frequency behavior** and eliminates steady-state error

For stability (Routh-Hurwitz criterion on the cubic), all coefficients must be positive and $(c + K_d)(k + K_p) > m K_i$.

### Why the Derivative Term Amplifies Noise

In the frequency domain, the derivative term has transfer function $K_d s$, which has **magnitude** $|K_d s| = K_d \omega$ at frequency $\omega$. This is a gain that **grows linearly** with frequency — high-frequency noise at $\omega_{\text{noise}}$ is amplified by $K_d \omega_{\text{noise}}$.

**Filtered derivative.** Replace $K_d s$ with a **first-order low-pass filtered derivative**:

$$D_f(s) = \frac{K_d s}{1 + \tau_f s}$$

where $\tau_f$ is the filter time constant. The magnitude is:

$$|D_f(j\omega)| = \frac{K_d \omega}{\sqrt{1 + \tau_f^2 \omega^2}}$$

For $\omega \ll 1/\tau_f$: $|D_f| \approx K_d \omega$ (behaves like pure derivative). For $\omega \gg 1/\tau_f$: $|D_f| \approx K_d / \tau_f$ (constant — noise amplification is bounded). Typical choice: $\tau_f = K_d / (N \cdot K_p)$ with $N \in [3, 20]$.

</details>

In [ ]:
from lib.pid_viz import show_pid_step_response_interactive
show_pid_step_response_interactive()

In [ ]:
from lib.pid_viz import show_pid_gain_effects
show_pid_gain_effects()

## Tuning: Ziegler-Nichols Method <a id="ziegler-nichols"></a>

A classical recipe (1942):

1. Set $K_i = K_d = 0$
2. Increase $K_p$ until the system exhibits **sustained oscillation** at gain $K_u$ with period $T_u$
3. Set PID gains from the table:

| Controller | $K_p$ | $K_i$ | $K_d$ |
|-----------|-------|-------|-------|
| P | $0.5 K_u$ | — | — |
| PI | $0.45 K_u$ | $0.54 K_u / T_u$ | — |
| PID | $0.6 K_u$ | $1.2 K_u / T_u$ | $0.075 K_u T_u$ |

This is a **quick starting point**, not an optimal tuning. Modern practice often uses simulation-based search or loop-shaping.

In [ ]:
from lib.pid_viz import show_ziegler_nichols_demo
show_ziegler_nichols_demo()

## Interactive: PID on CartPole <a id="pid-cartpole-interactive"></a>

In [ ]:
from lib.pid_viz import show_pid_cartpole_interactive
show_pid_cartpole_interactive()

> **Quick Check:** You increase $K_d$ on the CartPole PID and the response gets noisy. Why? What would you do in practice?

---

## PID Balancing <a id="pid-balancing"></a>

Implement a **discrete PID controller** that balances the pole from a small initial perturbation.

The controller should act on the **angle error**: $e = 0 - \theta$ (we want $\theta = 0$).

The `PIDController` class is pre-implemented in `lib/pid_controller.py` with:
- Anti-windup on the integral term
- Filtered derivative (to reduce noise sensitivity)
- Output clamping to $[-f_{\max}, f_{\max}]$

**Your task:** find gains $(K_p, K_i, K_d)$ that successfully balance the pole from $\theta_0 = 0.15$ rad (~8.6°).

In [ ]:
from lib.pid_controller import PIDController

# TODO: tune these gains!
pid = PIDController(Kp=40.0, Ki=0.0, Kd=15.0, dt=env.dt, x_setpoint=TARGET_X, Kp_x=1.0)

In [ ]:
pid.reset()
initial_state_pid = np.array([0.0, 0.15, 0.0, 0.0])
ts_pid, states_pid, controls_pid, rewards_pid = env.run_episode(
    pid, T=5.0, initial_state=initial_state_pid
)
plot_episode(ts_pid, states_pid, controls_pid, rewards_pid, title="PID Balancing", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_pid, states_pid, controls_pid, env)

**Questions to explore:**
- What happens if you increase $K_p$ too much?
- What happens if you set $K_d = 0$?
- Can PID balance from $\theta_0 = 0.5$ rad (~28.6°)? From $\theta_0 = 1.0$ rad?
- Try adding $K_i > 0$. Does it help?

---
# 6. LQR — Optimal Control via Bellman's Principle <a id="lqr"></a>

PID gives us a *reasonable* controller, but is it the *best*? LQR answers: **given a quadratic cost function, what is the optimal linear feedback?**

## Problem Statement <a id="lqr-problem"></a>

Minimise the **infinite-horizon quadratic cost:**

$$J = \int_0^\infty \left( x^\top Q x + u^\top R u \right) dt$$

subject to the linear dynamics $\dot{x} = Ax + Bu$.

- $Q \succeq 0$ penalises state deviation (e.g., large $Q_{\theta\theta}$ penalises angle error)
- $R \succ 0$ penalises control effort (larger $R$ → gentler but slower control)

## Deriving LQR from the HJB Equation <a id="lqr-hjb"></a>

Define the **value function** (cost-to-go from state $x$):

$$V(x) = \min_{u(\cdot)} \int_0^\infty \left(x^\top Q x + u^\top R u\right) dt$$

By **Bellman's principle**, $V$ satisfies the **Hamilton-Jacobi-Bellman (HJB)** equation:

$$0 = \min_u \left[ x^\top Q x + u^\top R u + \nabla V^\top (Ax + Bu) \right]$$

**Step 1:** Take the derivative w.r.t. $u$ and set to zero:

$$2Ru + B^\top \nabla V = 0 \quad \Rightarrow \quad u^* = -\frac{1}{2}R^{-1}B^\top \nabla V$$

**Step 2:** Guess a quadratic value function $V(x) = x^\top P x$, so $\nabla V = 2Px$:

$$u^* = -R^{-1}B^\top P x = -Kx, \qquad K = R^{-1}B^\top P$$

**Step 3:** Substitute back into HJB to get the **Algebraic Riccati Equation (ARE):**

$$A^\top P + PA - PBR^{-1}B^\top P + Q = 0$$

## Intuition <a id="lqr-intuition"></a>

- $V(x) = x^\top P x$ is the optimal **cost-to-go** — it measures how "expensive" a state is
- $P$ simultaneously defines a **quadratic Lyapunov function** (connects to Section 8!)
- The optimal control is **linear** in the state: $u^* = -Kx$ — this is remarkable and a direct consequence of quadratic cost + linear dynamics
- Larger $Q$ → $P$ grows → controller works harder to bring state to zero
- Larger $R$ → $P$ shrinks → controller uses less force

## Finite-Horizon LQR: Riccati Backward Integration <a id="riccati"></a>

For a **finite** time horizon $[0, T]$, the Riccati equation becomes a differential equation integrated **backward** in time:

$$-\dot{P}(t) = A^\top P + PA - PBR^{-1}B^\top P + Q, \qquad P(T) = Q_f$$

As $T \to \infty$, $P(t) \to P_{\infty}$ (the solution of the ARE). This is **dynamic programming in action** — we solve from the end backward.

<details>
<summary><b>Derivation: LQR via Pontryagin's Minimum Principle (click to expand)</b></summary>

The HJB approach above derives LQR "forward" from the value function. Here is the classical **co-state** (Pontryagin) derivation, which works "backward" from necessary conditions for optimality.

### The Hamiltonian

Define the **Hamiltonian** (not to be confused with the Hamiltonian in mechanics):

$$H(x, \lambda, u) = x^\top Q x + u^\top R u + \lambda^\top (Ax + Bu)$$

where $\lambda(t) \in \mathbb{R}^n$ is the **co-state** (adjoint / Lagrange multiplier for the dynamics constraint). The Pontryagin minimum principle states that the optimal trajectory $(x^*(t), u^*(t), \lambda^*(t))$ must satisfy:

**1. Optimality in $u$ (stationarity):**

$$\frac{\partial H}{\partial u} = 2Ru + B^\top \lambda = 0 \quad \Rightarrow \quad u^* = -\frac{1}{2} R^{-1} B^\top \lambda$$

**2. State equation:**

$$\dot{x} = \frac{\partial H}{\partial \lambda} = Ax + Bu$$

**3. Co-state equation:**

$$\dot{\lambda} = -\frac{\partial H}{\partial x} = -2Qx - A^\top \lambda$$

### The Hamiltonian System

Substituting $u^* = -\frac{1}{2}R^{-1}B^\top \lambda$ into the state equation gives $\dot{x} = Ax - \frac{1}{2}BR^{-1}B^\top \lambda$. Together with the co-state equation, this is a **linear system** in $2n$ dimensions:

$$\begin{bmatrix} \dot{x} \\ \dot{\lambda} \end{bmatrix} = \underbrace{\begin{bmatrix} A & -\frac{1}{2}BR^{-1}B^\top \\ -2Q & -A^\top \end{bmatrix}}_{\mathcal{H}} \begin{bmatrix} x \\ \lambda \end{bmatrix}$$

The matrix $\mathcal{H}$ is called the **Hamiltonian matrix**. It has a special structure: if $\mu$ is an eigenvalue of $\mathcal{H}$, then so is $-\mu$ (the eigenvalues come in pairs symmetric about the imaginary axis). For a stabilizable and detectable system, $\mathcal{H}$ has exactly $n$ eigenvalues with $\text{Re}(\mu) < 0$ and $n$ with $\text{Re}(\mu) > 0$.

### Recovering the Riccati Equation

**Ansatz:** Assume the co-state is a linear function of the state: $\lambda(t) = 2P(t)\, x(t)$ for some symmetric matrix $P(t)$.

Differentiate: $\dot{\lambda} = 2\dot{P}\, x + 2P\, \dot{x}$.

Substitute the state equation $\dot{x} = Ax - BR^{-1}B^\top P x$:

$$\dot{\lambda} = 2\dot{P}\, x + 2P(Ax - BR^{-1}B^\top P x) = 2(\dot{P} + PA - PBR^{-1}B^\top P)\, x$$

But the co-state equation says $\dot{\lambda} = -2Qx - A^\top (2Px) = -2(Q + A^\top P)\, x$.

Equating the two expressions for all $x$:

$$\dot{P} + PA - PBR^{-1}B^\top P = -(Q + A^\top P)$$

$$\Rightarrow \quad -\dot{P} = A^\top P + PA - PBR^{-1}B^\top P + Q$$

This is the **Riccati differential equation**, identical to the one obtained from HJB. For the infinite-horizon problem ($T \to \infty$), $\dot{P} \to 0$ and we recover the **Algebraic Riccati Equation (ARE)**.

### Hamiltonian Eigenspace Method

The ARE can be solved directly from $\mathcal{H}$. Let $\begin{bmatrix} X \\ \Lambda \end{bmatrix}$ be the matrix whose columns span the **stable eigenspace** of $\mathcal{H}$ (the $n$-dimensional invariant subspace corresponding to eigenvalues with $\text{Re}(\mu) < 0$). Then:

$$P = \Lambda\, X^{-1}$$

This is because on the stable manifold, $\lambda(t) = \Lambda X^{-1} x(t)$, and since the optimal trajectory must be bounded (stable), it lies on this manifold. The `scipy.linalg.solve_continuous_are` function uses a numerically robust variant of this approach (the Schur decomposition of $\mathcal{H}$).

### Existence and Uniqueness

The ARE $A^\top P + PA - PBR^{-1}B^\top P + Q = 0$ has a **unique positive semi-definite solution** $P$ if and only if:

1. $(A, B)$ is **stabilizable**: all unstable modes of $A$ are controllable through $B$ (the system can be stabilised by state feedback, even if some stable modes are not controllable)
2. $(A, Q^{1/2})$ is **detectable**: all unstable modes of $A$ are observable through $Q^{1/2}$ (the cost function "sees" all unstable directions)

**Interpretation.** Stabilizability ensures a finite-cost control exists. Detectability ensures the cost $\int x^\top Q x\, dt$ is infinite if an unstable mode is excited — so the optimal controller must stabilise it. Without detectability, an unstable mode might have zero cost, and $P$ would not be unique.

</details>

In [ ]:
from lib.lqr_viz import show_riccati_backward
show_riccati_backward()

## LQR on CartPole <a id="lqr-cartpole"></a>

In [ ]:
from lib.lqr_viz import show_lqr_cartpole
show_lqr_cartpole(R_val=0.0001)

## Interactive: Effect of Q and R <a id="qr-effects"></a>

In [ ]:
from lib.lqr_viz import show_lqr_qr_effects_interactive
show_lqr_qr_effects_interactive()

## LQR vs PID <a id="lqr-vs-pid"></a>

In [ ]:
from lib.lqr_viz import show_lqr_vs_pid
show_lqr_vs_pid()

> **Quick Check:** LQR gives the optimal controller for a given $(Q, R)$. But who chooses $Q$ and $R$? Is this truly "optimal"?

---

## LQR Balancing <a id="lqr-balancing"></a>

LQR finds the **optimal** linear feedback $u = -Kx$ for the linearised system.

The `LQRController` solves the continuous Algebraic Riccati Equation and returns the gain matrix $K$.

**Your task:** create an LQR controller and compare its performance with PID on the same initial condition.

In [ ]:
from lib.lqr_controller import LQRController

# Q penalises state deviation, R penalises control effort
# TODO: experiment with different Q and R values
lqr = LQRController(env_or_AB=env, Q_diag=(1, 10, 1, 10), R_val=1.0, target_state=env.target_state)
print(f"LQR gain K = {lqr.K}")

In [ ]:
initial_state_lqr = np.array([0.0, 0.15, 0.0, 0.0])
ts_lqr, states_lqr, controls_lqr, rewards_lqr = env.run_episode(
    lqr, T=5.0, initial_state=initial_state_lqr
)
plot_episode(ts_lqr, states_lqr, controls_lqr, rewards_lqr, title="LQR Balancing", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_lqr, states_lqr, controls_lqr, env)

### PID vs LQR Comparison <a id="pid-vs-lqr"></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ts_pid, states_pid[:, 1], label='PID')
axes[0].plot(ts_lqr, states_lqr[:, 1], label='LQR')
axes[0].set_xlabel('t [s]'); axes[0].set_ylabel('θ [rad]')
axes[0].set_title('Angle comparison')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ts_pid, controls_pid, label='PID')
axes[1].plot(ts_lqr, controls_lqr, label='LQR')
axes[1].set_xlabel('t [s]'); axes[1].set_ylabel('u [N]')
axes[1].set_title('Control comparison')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

# Compute and compare total cost
Q = np.diag([1, 10, 1, 10]); R_val = 1.0
x_ref = env.target_state
cost_pid = np.sum([(s - x_ref) @ Q @ (s - x_ref) + R_val * u**2 for s, u in zip(states_pid, controls_pid)]) * env.dt
cost_lqr = np.sum([(s - x_ref) @ Q @ (s - x_ref) + R_val * u**2 for s, u in zip(states_lqr, controls_lqr)]) * env.dt
print(f"Quadratic cost J = ΣxᵀQx + uᵀRu:")
print(f"  PID: {cost_pid:.3f}")
print(f"  LQR: {cost_lqr:.3f}")

**Questions:**
- Why is LQR cost lower? Is this always the case?
- What happens when you increase Q_θ (penalise angle more)? What about increasing R?
- At what initial angle does LQR start to fail? (The linearisation assumption breaks.)

---
# 7. Model Predictive Control (MPC) <a id="mpc"></a>

LQR is optimal but assumes **no constraints**. In practice, actuators saturate, states must stay within bounds, and the planning horizon is finite.

**MPC** solves an **optimisation problem at every time step**, applying only the first control action, then re-solving — the **receding horizon** strategy.

## Optimisation Background <a id="optimisation-background"></a>

MPC requires solving a **Quadratic Program (QP)**:

$$\min_z \;\frac{1}{2} z^\top H z + c^\top z \quad \text{s.t.} \quad Az \le b$$

The **KKT conditions** (necessary & sufficient for convex QP) state that at the optimum:
1. **Stationarity:** $Hz^* + c + A^\top \lambda^* = 0$
2. **Primal feasibility:** $Az^* \le b$
3. **Dual feasibility:** $\lambda^* \ge 0$
4. **Complementary slackness:** $\lambda_i^*(A_i z^* - b_i) = 0$

<details>
<summary><b>Derivation: Lagrangian and KKT Conditions (click to expand)</b></summary>

### Constrained Optimisation via the Lagrangian

Consider the general problem: $\min_{z} f(z)$ subject to $g_i(z) \le 0$, $i = 1, \ldots, m$.

**The Lagrangian** is defined as

$$\mathcal{L}(z, \lambda) = f(z) + \sum_{i=1}^{m} \lambda_i\, g_i(z) = f(z) + \lambda^\top g(z)$$

where $\lambda_i \ge 0$ are the **Lagrange multipliers** (dual variables). The idea: penalise constraint violations by adding them to the objective with non-negative weights.

### Deriving the KKT Conditions

The **Karush-Kuhn-Tucker (KKT)** conditions are necessary conditions for a local minimum of a constrained problem (under mild regularity conditions). They follow from the geometry of constrained optimisation:

**1. Stationarity:** $\nabla_z \mathcal{L} = \nabla f(z^*) + \sum_{i=1}^{m} \lambda_i^* \nabla g_i(z^*) = 0$

At the optimum, $-\nabla f$ must lie in the cone spanned by the constraint normals $\nabla g_i$ of the **active** constraints. Otherwise, there exists a feasible descent direction.

**2. Primal feasibility:** $g_i(z^*) \le 0$ for all $i$

The optimal point must satisfy all constraints.

**3. Dual feasibility:** $\lambda_i^* \ge 0$ for all $i$

The multipliers are non-negative because inequality constraints can only "push" the solution, not "pull" it. A constraint $g_i \le 0$ that is active acts as a barrier; $\lambda_i > 0$ means the constraint is binding and affects the solution.

**4. Complementary slackness:** $\lambda_i^* g_i(z^*) = 0$ for all $i$

Since $\lambda_i^* \ge 0$ and $g_i(z^*) \le 0$, their product $\lambda_i^* g_i(z^*)$ is non-positive. It equals zero iff either the constraint is inactive ($g_i(z^*) < 0$ so $\lambda_i^* = 0$) or the constraint is active ($g_i(z^*) = 0$ and $\lambda_i^* \ge 0$). An inactive constraint does not influence the solution — its multiplier is zero.

### Convexity and Sufficiency

For a **convex** problem (convex $f$, convex $g_i$), KKT conditions are not just necessary but also **sufficient** for global optimality:

**Proof sketch.** At a KKT point $(z^*, \lambda^*)$, consider any feasible $z$. Since $f$ is convex:

$$f(z) \ge f(z^*) + \nabla f(z^*)^\top (z - z^*)$$

By stationarity: $\nabla f(z^*) = -\sum_i \lambda_i^* \nabla g_i(z^*)$, so:

$$f(z) \ge f(z^*) - \sum_i \lambda_i^* \nabla g_i(z^*)^\top (z - z^*)$$

Since each $g_i$ is convex: $g_i(z) \ge g_i(z^*) + \nabla g_i(z^*)^\top (z - z^*)$, so $\nabla g_i(z^*)^\top (z - z^*) \le g_i(z) - g_i(z^*)$. Therefore:

$$f(z) \ge f(z^*) - \sum_i \lambda_i^* \bigl(g_i(z) - g_i(z^*)\bigr) = f(z^*) - \sum_i \lambda_i^* g_i(z) + \underbrace{\sum_i \lambda_i^* g_i(z^*)}_{= 0 \text{ (comp. slack.)}}$$

Since $\lambda_i^* \ge 0$ and $g_i(z) \le 0$ (feasibility of $z$): $-\sum_i \lambda_i^* g_i(z) \ge 0$. Thus $f(z) \ge f(z^*)$ for all feasible $z$. $\square$

**Application to MPC.** The MPC cost is quadratic in $z = (u_0, \ldots, u_{N-1})$ (after eliminating states), the dynamics constraints are linear (equality, handled via substitution), and the control bounds $u_{\min} \le u_k \le u_{\max}$ are linear inequalities. So the MPC QP is convex, and the KKT conditions characterise the unique global optimum.

</details>

In [ ]:
from lib.optim_viz import show_constrained_optimization
show_constrained_optimization()

In [ ]:
from lib.optim_viz import show_qp_example
show_qp_example()

## MPC Formulation <a id="mpc-formulation"></a>

Given discretised dynamics $x_{k+1} = A_d x_k + B_d u_k$ and prediction horizon $N$:

$$\min_{u_0, \ldots, u_{N-1}} \;\sum_{k=0}^{N-1} \left(x_k^\top Q x_k + u_k^\top R u_k\right) + x_N^\top Q_f x_N$$

subject to: $x_{k+1} = A_d x_k + B_d u_k$, $u_{\min} \le u_k \le u_{\max}$

At each real time step:
1. Measure current state $x_0$
2. Solve the QP → get $u_0^*, u_1^*, \ldots, u_{N-1}^*$
3. Apply **only** $u_0^*$ to the real system
4. Shift the horizon forward and repeat

**Key insight:** MPC is solving a **truncated Bellman/dynamic programming** problem **online** at each step. Without constraints, MPC reduces to **finite-horizon LQR**.

<details>
<summary><b>Derivation: Dense QP Formulation, Stability, and Discretisation (click to expand)</b></summary>

### Reformulation as a Dense QP

The MPC problem has decision variables $(u_0, \ldots, u_{N-1})$ and implicit states $(x_1, \ldots, x_N)$ linked by $x_{k+1} = A_d x_k + B_d u_k$. We can **eliminate the states** by recursively substituting:

$$x_1 = A_d x_0 + B_d u_0$$
$$x_2 = A_d x_1 + B_d u_1 = A_d^2 x_0 + A_d B_d u_0 + B_d u_1$$
$$\vdots$$
$$x_k = A_d^k x_0 + \sum_{j=0}^{k-1} A_d^{k-1-j} B_d\, u_j$$

Stacking all states $\mathbf{x} = [x_1; \ldots; x_N] \in \mathbb{R}^{Nn}$ and controls $\mathbf{u} = [u_0; \ldots; u_{N-1}] \in \mathbb{R}^{Nm}$:

$$\mathbf{x} = \underbrace{\begin{bmatrix} A_d \\ A_d^2 \\ \vdots \\ A_d^N \end{bmatrix}}_{\mathbf{A}} x_0 + \underbrace{\begin{bmatrix} B_d & 0 & \cdots & 0 \\ A_d B_d & B_d & \cdots & 0 \\ \vdots & & \ddots & \vdots \\ A_d^{N-1}B_d & A_d^{N-2}B_d & \cdots & B_d \end{bmatrix}}_{\mathbf{B}}\, \mathbf{u}$$

Substituting into the cost $J = \sum_{k=0}^{N-1}(x_k^\top Q x_k + u_k^\top R u_k) + x_N^\top Q_f x_N$ and collecting terms:

$$J = \mathbf{u}^\top \underbrace{(\mathbf{B}^\top \bar{Q} \mathbf{B} + \bar{R})}_{H}\, \mathbf{u} + 2\underbrace{(\mathbf{A} x_0)^\top \bar{Q}\, \mathbf{B}}_{c^\top}\, \mathbf{u} + \text{const}$$

where $\bar{Q} = \text{blockdiag}(Q, \ldots, Q, Q_f)$ and $\bar{R} = \text{blockdiag}(R, \ldots, R)$. The box constraints $u_{\min} \le u_k \le u_{\max}$ become $\mathbf{u}_{\min} \le \mathbf{u} \le \mathbf{u}_{\max}$. This is a standard QP: $\min_{\mathbf{u}} \frac{1}{2}\mathbf{u}^\top H \mathbf{u} + c^\top \mathbf{u}$ s.t. linear bounds.

### Stability via Terminal Cost

A natural question: does the receding-horizon MPC closed-loop system converge to the origin?

**Theorem (Stability with terminal cost).** If $Q_f = P_{\text{LQR}}$ (the solution of the ARE from Section 6), then the MPC value function $V_N(x) = J^*(x)$ is a **Lyapunov function** for the closed-loop system.

**Proof sketch.** At time $k$, MPC solves the problem and obtains the optimal sequence $u_0^*, u_1^*, \ldots, u_{N-1}^*$ with cost $V_N(x_k)$. At time $k+1$, consider the **suboptimal** (but feasible) sequence: $u_1^*, \ldots, u_{N-1}^*, u_{\text{LQR}}(x_N^*)$ — shift the old sequence and append the LQR control. Its cost is:

$$\tilde{J} = V_N(x_k) - x_k^\top Q x_k - (u_0^*)^\top R u_0^* + (x_N^*)^\top Q x_N^* + u_{\text{LQR}}^\top R u_{\text{LQR}} + (x_{N+1}')^\top Q_f x_{N+1}' - (x_N^*)^\top Q_f x_N^*$$

Since $Q_f = P_{\text{LQR}}$ satisfies the ARE, the LQR tail terms cancel: $(x_N^*)^\top Q x_N^* + u_{\text{LQR}}^\top R u_{\text{LQR}} + (x_{N+1}')^\top P (x_{N+1}') - (x_N^*)^\top P x_N^* = 0$ (this is exactly what the ARE guarantees). So:

$$\tilde{J} = V_N(x_k) - x_k^\top Q x_k - (u_0^*)^\top R u_0^*$$

Since the optimal cost at $x_{k+1}$ satisfies $V_N(x_{k+1}) \le \tilde{J}$ (optimality beats the suboptimal sequence):

$$V_N(x_{k+1}) - V_N(x_k) \le -(x_k^\top Q x_k + (u_0^*)^\top R u_0^*) < 0 \quad \text{for } x_k \neq 0$$

So $V_N$ is strictly decreasing along trajectories — it is a Lyapunov function, and the system is asymptotically stable. $\square$

**Remark.** With $Q_f = 0$, the LQR tail terms do **not** cancel, and MPC can be unstable for short horizons $N$.

### Exact Discretisation (Zero-Order Hold)

The MPC formulation uses discrete-time dynamics $x_{k+1} = A_d x_k + B_d u_k$. These come from **exact discretisation** of $\dot{x} = Ax + Bu$ assuming $u$ is held constant over $[k\Delta t, (k+1)\Delta t)$ (zero-order hold):

$$x(t + \Delta t) = e^{A\Delta t} x(t) + \left(\int_0^{\Delta t} e^{A\tau}\, d\tau\right) B\, u$$

**Derivation.** The solution of $\dot{x} = Ax + Bu$ with constant $u$ over $[0, \Delta t]$ is:

$$x(\Delta t) = e^{A\Delta t} x(0) + \int_0^{\Delta t} e^{A(\Delta t - \tau)} B\, u\, d\tau = e^{A\Delta t} x(0) + \left(\int_0^{\Delta t} e^{A\sigma}\, d\sigma\right) B\, u$$

where the substitution $\sigma = \Delta t - \tau$ was used and $u$ was pulled out of the integral (constant). So:

$$A_d = e^{A\Delta t}, \qquad B_d = \left(\int_0^{\Delta t} e^{A\tau}\, d\tau\right) B$$

When $A$ is invertible, $\int_0^{\Delta t} e^{A\tau}\, d\tau = A^{-1}(e^{A\Delta t} - I) = A^{-1}(A_d - I)$, giving the closed-form $B_d = A^{-1}(A_d - I) B$.

**Comparison with Euler discretisation.** Forward Euler $x_{k+1} = (I + A\Delta t) x_k + B\Delta t\, u_k$ is the first-order Taylor approximation of $e^{A\Delta t} \approx I + A\Delta t$ and $\int_0^{\Delta t} e^{A\tau} d\tau \approx I\Delta t$. It introduces discretisation error of order $O(\Delta t^2)$ per step, which accumulates as $O(\Delta t)$ globally. The exact ZOH discretisation has **no discretisation error** (it is exact for constant $u$).

</details>

In [ ]:
from lib.mpc_viz import show_mpc_concept
show_mpc_concept()

## MPC on CartPole <a id="mpc-cartpole"></a>

The thin dashed lines show the **predicted trajectory** at every 10th step — you can see the horizon shifting forward:

In [ ]:
from lib.mpc_viz import show_mpc_cartpole
show_mpc_cartpole()

## Effect of Control Constraints <a id="mpc-constraints"></a>

What happens when we tighten the force limits?

In [ ]:
from lib.mpc_viz import show_mpc_constrained_vs_unconstrained
show_mpc_constrained_vs_unconstrained()

> **Quick Check:** Why is MPC computationally expensive compared to LQR? When is the extra cost justified?

---

## MPC Swing-Up (Stretch) <a id="mpc-swingup"></a>

MPC solves a constrained optimisation problem at every step.

### Linear MPC

The `MPCController` uses **linearised** dynamics and `scipy.optimize.minimize` (SLSQP) to solve the QP. Because it uses a linear model, it struggles with true swing-up from $\theta = \pi$. It works well for larger-angle balancing that PID/LQR can't handle.

In [ ]:
from lib.mpc_controller import MPCController

mpc = MPCController(
    env=env,
    N=30,
    Q_diag=(10, 100, 10, 100),
    R_val=0.005,
    Qf_scale=20.0,
    f_max=10.0,
    target_state=env.target_state,
)

In [ ]:
# MPC for large-angle balancing (where LQR alone starts to struggle)
initial_state_mpc = np.array([0.0, 0.5, 0.0, 0.0])  # ~28.6°
ts_mpc, states_mpc, controls_mpc, rewards_mpc = env.run_episode(
    mpc, T=3.0, initial_state=initial_state_mpc
)
plot_episode(ts_mpc, states_mpc, controls_mpc, rewards_mpc, title="MPC Balancing (θ₀=0.5 rad)", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_mpc, states_mpc, controls_mpc, env)

### Compare: LQR vs MPC with tight force constraints <a id="lqr-vs-mpc"></a>

In [ ]:
# Tight force constraint: f_max = 3N
env_tight = CartPoleEnv(f_max=3.0, target_x=TARGET_X)
lqr_tight = LQRController(env_or_AB=env_tight, f_max=3.0, target_state=env_tight.target_state)
mpc_tight = MPCController(env=env_tight, f_max=3.0, target_state=env_tight.target_state)

init_tight = np.array([0.0, 0.3, 0.0, 0.0])

ts_lt, st_lt, u_lt, _ = env_tight.run_episode(lqr_tight, T=5.0, initial_state=init_tight)
ts_mt, st_mt, u_mt, _ = env_tight.run_episode(mpc_tight, T=5.0, initial_state=init_tight)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ts_lt, st_lt[:, 1], label='LQR'); axes[0].plot(ts_mt, st_mt[:, 1], label='MPC')
axes[0].set_xlabel('t [s]'); axes[0].set_ylabel('θ [rad]')
axes[0].set_title('Angle (f_max=3 N)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ts_lt, u_lt, label='LQR'); axes[1].plot(ts_mt, u_mt, label='MPC')
axes[1].axhline(3, color='r', ls=':', alpha=0.5); axes[1].axhline(-3, color='r', ls=':', alpha=0.5)
axes[1].set_xlabel('t [s]'); axes[1].set_ylabel('u [N]')
axes[1].set_title('Control (f_max=3 N)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

In [ ]:
animate_episode(ts_mt, st_mt, u_mt, env_tight)

**Questions:**
- MPC respects the force limit by design. How does LQR handle it? (Hint: look at the clipping.)
- What happens to MPC performance when you decrease N (shorter horizon)?
- Why is MPC much slower to run than LQR?

### Non-Linear MPC Swing-Up <a id="nonlinear-mpc-swingup"></a>

The `NonlinearMPCController` uses the **full nonlinear dynamics** for prediction (RK4 rollout), enabling true swing-up from $\theta \approx \pi$. It solves an NLP at each step, so it is slower than linear MPC, but handles the full range of motion.

In [ ]:
from lib.mpc_controller import NonlinearMPCController

nlmpc = NonlinearMPCController(
    env=env,
    N=50,
    Q_diag=(10, 100, 10, 100),
    R_val=0.005,
    Qf_scale=100.0,
    f_max=10.0,
    target_state=env.target_state,
)

In [ ]:
nlmpc.reset()
initial_state_nlmpc = np.array([0.0, np.pi - 0.05, 0.0, 0.0])
ts_nlmpc, states_nlmpc, controls_nlmpc, rewards_nlmpc = env.run_episode(
    nlmpc, T=6.0, initial_state=initial_state_nlmpc
)
plot_episode(ts_nlmpc, states_nlmpc, controls_nlmpc, rewards_nlmpc, title="Non-Linear MPC Swing-Up", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_nlmpc, states_nlmpc, controls_nlmpc, env)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ts_sw, states_sw[:, 1], label="Energy + LQR")
axes[0].plot(ts_nlmpc, states_nlmpc[:, 1], label="Non-Linear MPC")
axes[0].set_xlabel("t [s]")
axes[0].set_ylabel("θ [rad]")
axes[0].set_title("Swing-up: Energy vs Non-Linear MPC")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts_sw, controls_sw, label="Energy + LQR")
axes[1].plot(ts_nlmpc, controls_nlmpc, label="Non-Linear MPC")
axes[1].set_xlabel("t [s]")
axes[1].set_ylabel("u [N]")
axes[1].set_title("Control")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

---
# 8. Lyapunov Stability & Energy-Based Control <a id="lyapunov"></a>

All the methods above rely on **linearisation** — they work near the equilibrium but fail for large deviations (e.g., swinging the cart-pole *up* from the bottom).

**Lyapunov theory** gives us tools to reason about stability of **nonlinear** systems.

## Lyapunov's Direct Method <a id="lyapunov-direct"></a>

Given $\dot{x} = f(x, u)$ with equilibrium at $x = 0$, if there exists a function $V(x)$ such that:

1. $V(0) = 0$ and $V(x) > 0$ for all $x \ne 0$ (positive definite)
2. $\dot{V}(x) = \nabla V^\top f(x, u) \le 0$ (non-increasing along trajectories)

then the equilibrium is **stable**. If $\dot{V} < 0$ (strictly), it's **asymptotically stable**.

optimal $u = \arg\min_u \dot{V}(x, u) = \arg\min_u \nabla V^\top f(x, u)$

## Connection to LQR <a id="lyapunov-lqr"></a>

For the LQR-controlled system $\dot{x} = (A - BK)x$, the function $V(x) = x^\top P x$ where $P$ solves the ARE is a **Lyapunov function**:

$$\dot{V} = x^\top \left[(A-BK)^\top P + P(A-BK)\right] x = -x^\top (Q + K^\top R K) x < 0$$

So LQR simultaneously provides both the optimal controller **and** a certificate of stability.

<details>
<summary><b>Derivation: Lyapunov's Direct Method and La Salle's Principle (click to expand)</b></summary>

### Proof Sketch: Lyapunov's Stability Theorem

**Theorem.** If there exists $V: \mathbb{R}^n \to \mathbb{R}$ with (i) $V(0) = 0$, (ii) $V(x) > 0$ for $x \neq 0$, (iii) $V(x) \to \infty$ as $\|x\| \to \infty$ (radially unbounded), and (iv) $\dot{V}(x) = \nabla V(x)^\top f(x) < 0$ for all $x \neq 0$, then $x = 0$ is **globally asymptotically stable**.

**Proof sketch.**

**Step 1 (Boundedness).** Take any initial condition $x_0$. The sublevel set $\Omega_c = \{x : V(x) \le c\}$ with $c = V(x_0)$ contains $x_0$. Since $V$ is continuous, positive definite, and radially unbounded, $\Omega_c$ is closed and bounded (compact). Since $\dot{V} < 0$, $V(x(t))$ is strictly decreasing, so $V(x(t)) \le V(x_0) = c$ for all $t \ge 0$. Therefore $x(t) \in \Omega_c$ for all $t \ge 0$ — the trajectory is bounded.

**Step 2 (Monotone convergence).** $V(x(t))$ is a continuous function that is strictly decreasing and bounded below (by $0$). By the monotone convergence theorem, $V(x(t)) \to V_\infty \ge 0$ as $t \to \infty$.

**Step 3 (Convergence to origin).** Suppose $V_\infty > 0$. Then $x(t)$ stays in the compact set $S = \{x : V_\infty \le V(x) \le c\}$ which does not contain the origin. On $S$, $\dot{V}$ is continuous and strictly negative, so $\dot{V}(x) \le -\alpha < 0$ for some $\alpha > 0$ (by compactness). Then $V(x(t)) \le V(x_0) - \alpha t \to -\infty$, contradicting $V \ge 0$. Therefore $V_\infty = 0$, and since $V$ is positive definite, $x(t) \to 0$. $\square$

**Remark.** Without radial unboundedness, one can only prove **local** asymptotic stability (within some sublevel set $\Omega_c$).

### La Salle's Invariance Principle

What if $\dot{V} \le 0$ (not strictly negative) — i.e. $\dot{V}(x) = 0$ is possible for some $x \neq 0$?

**Theorem (La Salle).** Let $\Omega_c = \{x : V(x) \le c\}$ be a compact positively invariant set and $\dot{V}(x) \le 0$ on $\Omega_c$. Define $E = \{x \in \Omega_c : \dot{V}(x) = 0\}$. Let $M$ be the **largest invariant set** contained in $E$. Then every trajectory starting in $\Omega_c$ converges to $M$ as $t \to \infty$.

**Proof sketch.** By the same monotone convergence argument, $V(x(t)) \to V_\infty$. The $\omega$-limit set $\omega(x_0) = \bigcap_{T \ge 0} \overline{\{x(t) : t \ge T\}}$ is nonempty (by compactness), invariant, and connected. On $\omega(x_0)$, $V$ is constant (it equals $V_\infty$), so $\dot{V} = 0$ on $\omega(x_0)$. Since $\omega(x_0)$ is invariant and $\dot{V} = 0$ there, we have $\omega(x_0) \subseteq M$. $\square$

**Application.** If the only invariant set in $E = \{\dot{V} = 0\}$ is the origin, then $M = \{0\}$ and the system is asymptotically stable — even though $\dot{V}$ is only negative semi-definite.

**Example.** For a damped pendulum $\ddot{\theta} + b\dot{\theta} + \sin\theta = 0$ with $V = \frac{1}{2}\dot{\theta}^2 + (1 - \cos\theta)$: $\dot{V} = -b\dot{\theta}^2 \le 0$. The set $E = \{\dot{\theta} = 0\}$ contains all points $(\theta, 0)$. But the only invariant set in $E$ is the equilibrium $(\theta, \dot{\theta}) = (0, 0)$ (because $\dot{\theta} = 0$ implies $\ddot{\theta} = -\sin\theta$, and $\ddot{\theta} = 0$ requires $\sin\theta = 0$, and the stable equilibrium is $\theta = 0$). By La Salle, the origin is asymptotically stable.

</details>

In [ ]:
from lib.lyapunov_viz import show_lyapunov_concept
show_lyapunov_concept()

## Energy as a Lyapunov Function <a id="energy-lyapunov"></a>

For mechanical systems, **total energy** is a natural Lyapunov candidate.

For the cart-pole, the total energy is:

$$E = \frac{1}{2}m_c \dot{x}^2 + \frac{1}{2}m_p\left[(\dot{x} + l\dot{\theta}\cos\theta)^2 + (l\dot{\theta}\sin\theta)^2\right] + m_p g l \cos\theta$$

At the **upright equilibrium**: $E_{\text{up}} = m_p g l$.

The energy landscape reveals the structure of the system:

In [ ]:
from lib.lyapunov_viz import show_energy_landscape
show_energy_landscape()

## Energy-Based Swing-Up <a id="energy-swingup"></a>

**Idea:** pump energy into the system until $E \approx E_{\text{up}}$, then switch to LQR for stabilisation.

Energy-pumping controller:

$$u = k_E (E - E_{\text{up}}) \, \dot{\theta} \cos\theta \;-\; k_x x \;-\; k_{\dot x} \dot{x}$$

- The first term pumps/removes energy: $\dot{\theta}\cos\theta$ determines the direction that adds energy to the pendulum
- The $k_x, k_{\dot x}$ terms keep the cart centered (prevent drift)
- When $E \approx E_{\text{up}}$: the energy term vanishes smoothly
- Near upright ($|\theta| < \theta_{\text{switch}}$): switch to LQR for stabilisation

<details>
<summary><b>Derivation: Energy-Pumping Controller from Lyapunov Theory (click to expand)</b></summary>

### Computing $\dot{E}$ — How Control Input Affects Energy

The cart-pole equations of motion (from the Euler-Lagrange derivation in Section 4) can be written abstractly as:

$$(m_c + m_p)\ddot{x} + m_p l \ddot{\theta}\cos\theta - m_p l \dot{\theta}^2 \sin\theta = u$$
$$m_p l \ddot{x}\cos\theta + m_p l^2 \ddot{\theta} - m_p g l \sin\theta = 0$$

The total energy is $E = T + V$ (kinetic + potential). For a cart-pole, the power delivered by the control force $u$ is $P = u \dot{x}$ (force times velocity). By the work-energy theorem:

$$\frac{dE}{dt} = u\,\dot{x}$$

The **pendulum's energy** (excluding the cart's kinetic energy $\frac{1}{2}m_c \dot{x}^2$) is more relevant for swing-up:

$$E_p = \frac{1}{2}m_p l^2 \dot{\theta}^2 + m_p g l (1 - \cos\theta)$$

(where we shifted the potential so $E_p = 0$ at the bottom $\theta = \pi$ and $E_p^{\text{up}} = 2m_p g l$ at the top $\theta = 0$). To find $\dot{E}_p$, differentiate using the chain rule:

$$\dot{E}_p = m_p l^2 \dot{\theta}\ddot{\theta} + m_p g l \sin\theta\, \dot{\theta}$$

From the second EOM: $\ddot{\theta} = \frac{g}{l}\sin\theta - \frac{\ddot{x}}{l}\cos\theta$. Substituting:

$$\dot{E}_p = m_p l \dot{\theta}\left(l\ddot{\theta} + g\sin\theta\right) - m_p l \dot{\theta} g\sin\theta + m_p g l \sin\theta\, \dot{\theta}$$

After simplification (the $g\sin\theta$ terms cancel within the EOM substitution): $\dot{E}_p = -m_p l \ddot{x} \dot{\theta} \cos\theta$. Since the cart acceleration $\ddot{x}$ depends on $u$ through the first EOM, the key relation is:

$$\dot{E}_p \approx -\frac{u}{m_c + m_p}\, m_p l\, \dot{\theta}\cos\theta$$

(in the simplified case where we neglect the coupling terms). The essential structure is:

$$\dot{E}_p \propto -u\, \dot{\theta}\cos\theta$$

### Choosing $u$ to Control Energy

Define the energy error $\tilde{E} = E_p - E_p^{\text{up}}$ and the Lyapunov candidate:

$$W = \frac{1}{2}\tilde{E}^2$$

Then:

$$\dot{W} = \tilde{E}\, \dot{E}_p \propto -\tilde{E}\, u\, \dot{\theta}\cos\theta$$

We want $\dot{W} \le 0$ (energy error decreasing). Choose $u = k_E\, \tilde{E}\, \dot{\theta}\cos\theta$ with $k_E > 0$:

$$\dot{W} \propto -k_E\, \tilde{E}^2\, (\dot{\theta}\cos\theta)^2 \le 0$$

This is negative semi-definite. Equality $\dot{W} = 0$ occurs when $\tilde{E} = 0$ (goal reached) or $\dot{\theta}\cos\theta = 0$ (at $\dot{\theta} = 0$ or $\theta = \pm\pi/2$). By La Salle's invariance principle, the system converges to the largest invariant set where $\dot{W} = 0$, which generically is $\tilde{E} = 0$ (energy equals the target).

### Why the Controller Works

- When $E_p < E_p^{\text{up}}$ ($\tilde{E} < 0$): the controller injects energy by applying force **in the direction that increases the pendulum's swing**
- When $E_p > E_p^{\text{up}}$ ($\tilde{E} > 0$): the controller removes energy by **opposing** the swing
- The factor $\dot{\theta}\cos\theta$ ensures the force is applied at the moment of maximum energy transfer (when the pendulum passes through the bottom, $\theta \approx \pi$, $\cos\theta \approx -1$, $\dot{\theta}$ is maximal)

### Why Switching to LQR is Necessary

The energy controller drives $E_p \to E_p^{\text{up}}$, but it does **not** stabilise the upright equilibrium. At the target energy, the pendulum has the right total energy to be at the top, but it could be oscillating through the top with nonzero velocity. The energy controller produces sustained oscillations near the homoclinic orbit, not convergence to the fixed point. LQR provides the local stabilisation once the pendulum enters the region of attraction near $\theta = 0$.

</details>

In [ ]:
from lib.lyapunov_viz import show_energy_swingup_demo
show_energy_swingup_demo()

## Region of Attraction <a id="region-of-attraction"></a>

LQR only works in a region around the upright equilibrium. How large is this region?

In [ ]:
from lib.lyapunov_viz import show_region_of_attraction
show_region_of_attraction()

<details>
<summary><b>Advanced: Control Lyapunov Functions (click to expand)</b></summary>

A **Control Lyapunov Function (CLF)** is a Lyapunov function $V(x)$ for which there **exists** a control input $u$ making $\dot{V}(x, u) < 0$ for all $x \ne 0$.

**Sontag's universal formula** gives a constructive way to find such $u$:

$$u = -\frac{L_f V + \sqrt{(L_f V)^2 + (L_g V)^4}}{L_g V}$$

where $L_f V = \nabla V \cdot f(x)$ and $L_g V = \nabla V \cdot g(x)$ are Lie derivatives.

The energy-based swing-up controller is a practical instance of a CLF-like approach: we treat $(E - E_{\text{up}})^2$ as a Lyapunov function and choose $u$ to decrease it.

In modern robotics, CLF constraints are often combined with QP-based controllers (CLF-QP) to guarantee stability while satisfying other constraints.

</details>

> **Quick Check:** Why do we need a switching strategy (energy → LQR) rather than using energy control alone?

---

## Energy-Based Swing-Up <a id="energy-swingup"></a>

PID and LQR work near the upright position, but they **cannot** swing the pole up from the bottom ($\theta = \pi$).

The **energy-based** strategy:
1. Pump energy into the system until $E \approx E_{\text{upright}}$
2. When close to upright, switch to LQR for stabilisation

The controller is:

$$u = \begin{cases} k_E (E - E_{\text{up}}) \, \dot{\theta} \cos\theta \;-\; k_x x \;-\; k_{\dot{x}} \dot{x} & \text{if far from upright} \\ -Kx & \text{if near upright (LQR)} \end{cases}$$

The first term pumps energy (its magnitude vanishes as $E \to E_{\text{up}}$), while the $k_x, k_{\dot{x}}$ terms keep the cart centered.

The `EnergySwingUpController` is pre-implemented. **Your task:** understand its parameters and test it.

In [ ]:
from lib.energy_controller import EnergySwingUpController

energy_ctrl = EnergySwingUpController(
    env=env,
    k_energy=20.0,       # energy pumping gain
    k_x=1.0,             # cart centering (position)
    k_xd=1.0,            # cart centering (velocity)
    switch_theta=0.3,    # switch to LQR when |θ| < this [rad]
    switch_theta_dot=2.0, # and |θ̇| < this [rad/s]
    target_x=TARGET_X,
)

In [ ]:
initial_state_swing = np.array([0.0, np.pi - 0.05, 0.0, 0.0])  # near bottom
ts_sw, states_sw, controls_sw, rewards_sw = env.run_episode(
    energy_ctrl, T=10.0, initial_state=initial_state_swing
)
plot_episode(ts_sw, states_sw, controls_sw, rewards_sw, title="Energy-Based Swing-Up", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_sw, states_sw, controls_sw, env)

### Visualise energy convergence <a id="energy-convergence"></a>

In [ ]:
energies = np.array([env.energy(s) for s in states_sw])
E_target = env.upright_energy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ts_sw, energies, 'b', lw=1.5, label='E(t)')
ax1.axhline(E_target, color='r', ls='--', label=f'E_upright = {E_target:.3f}')
ax1.set_xlabel('t [s]'); ax1.set_ylabel('Energy')
ax1.set_title('Energy over time'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(states_sw[:, 1], states_sw[:, 3], 'b', lw=0.5)
ax2.plot(states_sw[0, 1], states_sw[0, 3], 'go', ms=8, label='start')
ax2.plot(0, 0, 'r*', ms=12, label='target')
ax2.set_xlabel('θ [rad]'); ax2.set_ylabel('θ̇ [rad/s]')
ax2.set_title('Phase portrait'); ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

**Experiment:**
- What happens if `k_energy` is too small? Too large?
- What if `switch_theta` is too small (LQR engages too late) or too large (switches too early)?
- Can you make it work faster?

---
# 9. Reinforcement Learning — Bellman Revisited <a id="rl"></a>

We close the loop on **Bellman's principle** by moving from model-based optimal control to **model-free** learning.

## From Optimal Control to RL <a id="optimal-to-rl"></a>

| | LQR / MPC | RL |
|---|---|---|
| Model | Known ($\dot{x} = Ax + Bu$) | Unknown (learn from data) |
| Bellman equation | Continuous HJB | Discrete Bellman |
| Solution | Algebraic (ARE) or online QP | Iterative (Q-learning, PG) |
| Optimality | Provably optimal (for given cost) | Approximately optimal |

## Markov Decision Process (MDP) <a id="mdp"></a>

An MDP is defined by $(\mathcal{S}, \mathcal{A}, P, r, \gamma)$:
- $\mathcal{S}$: state space, $\mathcal{A}$: action space
- $P(s'|s,a)$: transition probability (the "model" — unknown in model-free RL)
- $r(s,a)$: reward function
- $\gamma \in [0, 1)$: discount factor

## The Bellman Equation (Discrete-Time) <a id="bellman-discrete"></a>

**Value of a policy** $\pi$:

$$V^\pi(s) = \mathbb{E}_{\pi}\left[\sum_{t=0}^{\infty} \gamma^t r(s_t, a_t) \,\middle|\, s_0 = s\right]$$

**Bellman equation for $V^\pi$:**

$$V^\pi(s) = \mathbb{E}\left[r + \gamma V^\pi(s')\right]$$

**Bellman optimality equation:**

$$V^*(s) = \max_a \mathbb{E}\left[r + \gamma V^*(s')\right]$$

Compare with the continuous HJB from Section 6:

$$0 = \min_u \left[\ell(x,u) + \nabla V^\top f(x,u)\right]$$

The structure is identical: **find the value function, derive the optimal policy from it.**

<details>
<summary><b>Derivation: Bellman Equations and Convergence of Value Iteration (click to expand)</b></summary>

### Deriving the Bellman Expectation Equation

Start from the definition of $V^\pi$:

$$V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty} \gamma^t r(s_t, a_t) \,\middle|\, s_0 = s\right]$$

Split the sum into the first reward and the rest:

$$V^\pi(s) = \mathbb{E}_\pi\left[r(s_0, a_0) + \gamma \sum_{t=1}^{\infty} \gamma^{t-1} r(s_t, a_t) \,\middle|\, s_0 = s\right]$$

$$= \mathbb{E}_\pi\left[r(s_0, a_0) + \gamma \sum_{t'=0}^{\infty} \gamma^{t'} r(s_{t'+1}, a_{t'+1}) \,\middle|\, s_0 = s\right]$$

By the **Markov property**, the future trajectory from $s_1$ onward is independent of $s_0$ given $s_1$. So $\mathbb{E}_\pi\left[\sum_{t'=0}^{\infty} \gamma^{t'} r(s_{t'+1}, a_{t'+1}) \,\middle|\, s_0 = s, s_1 = s'\right] = V^\pi(s')$. Taking the expectation over $a_0 \sim \pi(\cdot|s)$ and $s_1 \sim P(\cdot|s, a_0)$:

$$V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a|s) \sum_{s' \in \mathcal{S}} P(s'|s, a)\left[r(s, a) + \gamma\, V^\pi(s')\right]$$

This is the **Bellman expectation equation** — a system of $|\mathcal{S}|$ linear equations in $|\mathcal{S}|$ unknowns.

### Bellman Optimality Equation

The **optimal value function** $V^*(s) = \max_\pi V^\pi(s)$ satisfies:

$$V^*(s) = \max_{a \in \mathcal{A}} \sum_{s'} P(s'|s, a)\left[r(s, a) + \gamma\, V^*(s')\right]$$

This follows because the optimal policy at state $s$ chooses the action maximising the immediate reward plus discounted future value. The optimal policy is **greedy** w.r.t. $V^*$: $\pi^*(s) = \arg\max_a \sum_{s'} P(s'|s,a)[r(s,a) + \gamma V^*(s')]$.

### Convergence of Value Iteration (Contraction Mapping)

**Value iteration** updates $V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)[r(s,a) + \gamma V_k(s')]$ for all $s$. Define the **Bellman optimality operator** $\mathcal{T}$:

$$(\mathcal{T}V)(s) = \max_a \sum_{s'} P(s'|s,a)\left[r(s,a) + \gamma V(s')\right]$$

**Theorem ($\mathcal{T}$ is a $\gamma$-contraction in $\|\cdot\|_\infty$).** For any $V, V': \mathcal{S} \to \mathbb{R}$:

$$\|\mathcal{T}V - \mathcal{T}V'\|_\infty \le \gamma \|V - V'\|_\infty$$

**Proof.** For any state $s$:

$$(\mathcal{T}V)(s) - (\mathcal{T}V')(s) = \max_a \sum_{s'} P(s'|s,a)[r + \gamma V(s')] - \max_a \sum_{s'} P(s'|s,a)[r + \gamma V'(s')]$$

Since $\max_a f(a) - \max_a g(a) \le \max_a [f(a) - g(a)]$:

$$(\mathcal{T}V)(s) - (\mathcal{T}V')(s) \le \max_a \sum_{s'} P(s'|s,a)\, \gamma\, [V(s') - V'(s')]$$

$$\le \max_a \sum_{s'} P(s'|s,a)\, \gamma\, \|V - V'\|_\infty = \gamma \|V - V'\|_\infty$$

By symmetry, $(\mathcal{T}V')(s) - (\mathcal{T}V)(s) \le \gamma \|V - V'\|_\infty$, so $|(\mathcal{T}V)(s) - (\mathcal{T}V')(s)| \le \gamma \|V - V'\|_\infty$ for all $s$. $\square$

**Banach fixed-point theorem.** Since $(\mathcal{S} \to \mathbb{R}, \|\cdot\|_\infty)$ is a complete metric space and $\mathcal{T}$ is a $\gamma$-contraction with $\gamma < 1$:

1. $\mathcal{T}$ has a **unique** fixed point $V^*$ (satisfying $\mathcal{T}V^* = V^*$, i.e. the Bellman optimality equation)
2. For any initial $V_0$, the iterates $V_{k+1} = \mathcal{T}V_k$ converge: $\|V_k - V^*\|_\infty \le \gamma^k \|V_0 - V^*\|_\infty$
3. The convergence rate is **geometric** with ratio $\gamma$

</details>

## Q-Function <a id="q-function"></a>

$$Q^*(s, a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a')\right]$$

Advantage: we can pick the best action without knowing the model: $a^* = \arg\max_a Q^*(s, a)$.

## Policy Gradient Methods <a id="policy-gradient"></a>

Instead of learning $V$ or $Q$, directly optimise a **parameterised policy** $\pi_\theta(a|s)$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the return.

**REINFORCE** uses this directly. **Actor-Critic** methods reduce variance by learning a baseline $V^\pi(s)$.

<details>
<summary><b>Derivation: Policy Gradient Theorem (click to expand)</b></summary>

### The Log-Derivative Trick

Define the **objective** as the expected return under policy $\pi_\theta$:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[R(\tau)\right] = \int p(\tau|\theta)\, R(\tau)\, d\tau$$

where $\tau = (s_0, a_0, s_1, a_1, \ldots)$ is a trajectory, $R(\tau) = \sum_{t=0}^T \gamma^t r(s_t, a_t)$, and $p(\tau|\theta) = p(s_0) \prod_{t=0}^T \pi_\theta(a_t|s_t)\, P(s_{t+1}|s_t, a_t)$ is the trajectory probability.

To compute $\nabla_\theta J$, we use the **log-derivative trick**: $\nabla_\theta p(\tau|\theta) = p(\tau|\theta)\, \nabla_\theta \log p(\tau|\theta)$. Then:

$$\nabla_\theta J = \int \nabla_\theta p(\tau|\theta)\, R(\tau)\, d\tau = \int p(\tau|\theta)\, \nabla_\theta \log p(\tau|\theta)\, R(\tau)\, d\tau = \mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log p(\tau|\theta)\, R(\tau)\right]$$

### Simplifying $\nabla_\theta \log p(\tau|\theta)$

Take the log of the trajectory probability:

$$\log p(\tau|\theta) = \log p(s_0) + \sum_{t=0}^{T} \log \pi_\theta(a_t|s_t) + \sum_{t=0}^{T} \log P(s_{t+1}|s_t, a_t)$$

The initial state distribution $p(s_0)$ and the transition probabilities $P(s'|s,a)$ do **not depend on $\theta$**, so their gradients vanish:

$$\nabla_\theta \log p(\tau|\theta) = \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)$$

This is remarkable: we can compute the policy gradient **without knowing the dynamics model** $P$.

### Rewards-to-Go (Causality)

Substituting back:

$$\nabla_\theta J = \mathbb{E}_{\tau}\left[\left(\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)\right) R(\tau)\right]$$

We can simplify further using **causality**: an action $a_t$ can only affect rewards at times $\ge t$. Formally, for $t' < t$: $\mathbb{E}[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot r_{t'}] = 0$ because $r_{t'}$ does not depend on $a_t$, and $\mathbb{E}_{a_t \sim \pi_\theta}[\nabla_\theta \log \pi_\theta(a_t|s_t)] = \nabla_\theta \sum_a \pi_\theta(a|s_t) = \nabla_\theta 1 = 0$. Therefore:

$$\nabla_\theta J = \mathbb{E}_\tau\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right], \qquad G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$$

where $G_t$ is the **reward-to-go** from time $t$.

### Baseline and Advantage

**Theorem.** Subtracting any state-dependent baseline $b(s_t)$ from $G_t$ does not change the expected gradient:

$$\mathbb{E}\left[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t)\right] = \mathbb{E}_{s_t}\left[b(s_t) \cdot \underbrace{\mathbb{E}_{a_t \sim \pi_\theta(\cdot|s_t)}\left[\nabla_\theta \log \pi_\theta(a_t|s_t)\right]}_{= \nabla_\theta \sum_a \pi_\theta(a|s_t) = 0}\right] = 0$$

**Proof.** The inner expectation equals $\sum_a \nabla_\theta \pi_\theta(a|s_t) = \nabla_\theta \sum_a \pi_\theta(a|s_t) = \nabla_\theta 1 = 0$. Since $b(s_t)$ depends only on $s_t$ (not on $a_t$), it can be pulled out of the inner expectation. $\square$

The optimal baseline (minimizing variance) is $b(s_t) = V^\pi(s_t)$. With this choice:

$$\nabla_\theta J = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A^\pi(s_t, a_t)\right]$$

where $A^\pi(s_t, a_t) = Q^\pi(s_t, a_t) - V^\pi(s_t)$ is the **advantage function** — how much better action $a_t$ is compared to the average action under $\pi$. This is the foundation of **Actor-Critic** methods: the **actor** updates $\theta$ using the policy gradient; the **critic** learns $V^\pi$ (or $Q^\pi$) to estimate the advantage and reduce variance.

</details>

## Unifying View <a id="unifying-view"></a>

| Method | Model? | Bellman | Online/Offline | Constraints? | Computation |
|--------|--------|--------|----------------|-------------|-------------|
| **PID** | No model | — | Online | No | Trivial |
| **LQR** | Linear model | HJB (ARE) | Offline | No | $O(n^3)$ once |
| **MPC** | Linear/nonlinear | Truncated DP, online | Online | Yes | QP per step |
| **RL** | No model | Discrete Bellman | Offline training | Reward shaping | GPU training |

> **Quick Check:** In the comparison table above, MPC uses the model online at every step, while RL learns offline. What are the trade-offs in terms of sample efficiency, safety, and generalization?

---

## RL Swing-Up (Demo) <a id="rl-swingup"></a>

Finally, let's see a **model-free** approach: REINFORCE (policy gradient).

A small neural network $\pi_\theta(s) \to u$ is trained by sampling episodes and updating weights to maximise expected return.

The training loop is pre-implemented. We'll train for a moderate number of episodes and inspect the result.

**Note:** This is a demo — RL typically requires much more tuning and training time to match classical controllers.

In [ ]:
from lib.rl_controller import REINFORCEController, train_reinforce, plot_training

rl_ctrl = REINFORCEController(
    action_bound=env.f_max,
    hidden=32,
    lr=1e-3,
    gamma=0.99,
)

# Training takes ~1-2 minutes
print("Training REINFORCE policy...")
reward_history = train_reinforce(env, rl_ctrl, n_episodes=300, T=5.0, print_every=50)

In [ ]:
plot_training(reward_history)
plt.show()

### Test the learned policy <a id="test-policy"></a>

In [ ]:
# Test from small perturbation (should be easier than swing-up)
rl_ctrl.reset()
ts_rl, states_rl, controls_rl, rewards_rl = env.run_episode(
    rl_ctrl, T=5.0, initial_state=np.array([0.0, 0.2, 0.0, 0.0])
)
plot_episode(ts_rl, states_rl, controls_rl, rewards_rl, title="RL (REINFORCE) — Balancing", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_rl, states_rl, controls_rl, env)

In [ ]:
# Test on swing-up (harder — may or may not work depending on training)
rl_ctrl.reset()
ts_rl2, states_rl2, controls_rl2, rewards_rl2 = env.run_episode(
    rl_ctrl, T=10.0, initial_state=np.array([0.0, np.pi, 0.0, 0.0])
)
plot_episode(ts_rl2, states_rl2, controls_rl2, rewards_rl2, title="RL (REINFORCE) — Swing-Up Attempt", target_x=TARGET_X)
plt.show()

In [ ]:
animate_episode(ts_rl2, states_rl2, controls_rl2, env)

### Reflect: Classical vs Learned Control <a id="reflect"></a>

| | Energy + LQR | RL (REINFORCE) |
|---|---|---|
| Model needed? | Yes (dynamics + linearisation) | No (learns from data) |
| Training | None (analytical) | Minutes to hours |
| Guarantees | Lyapunov stability | None |
| Generalization | Fixed to this system | Could adapt to new dynamics |

---
# 10. Summary & Comparisons <a id="summary"></a>

Let's compare the methods we've studied on the **same CartPole task**.

## Balancing: PID vs LQR vs MPC <a id="balancing-comparison"></a>

In [ ]:
from lib.cartpole_sim import simulate_cartpole, linearize_cartpole, solve_lqr, cartpole_dynamics
from lib.mpc_viz import discretize_linear, solve_mpc

theta0 = 0.3
state0 = np.array([0.0, theta0, 0.0, 0.0])
A, B = linearize_cartpole()
Q = np.diag([1, 10, 1, 10]); R = np.array([[1.0]])
K_lqr, _ = solve_lqr(A, B, Q, R)
dt = 0.02; T = 5.0

# PID
_int = [0.0]; _prev = [0.0]
def pid_ctrl(t, s):
    e = -s[1]; _int[0] += e*dt; _int[0] = np.clip(_int[0], -5, 5)
    de = (e - _prev[0])/dt; _prev[0] = e
    return 50*e + 0*_int[0] + 20*de
ts_p, st_p, u_p = simulate_cartpole(state0.copy(), pid_ctrl, T, dt)

# LQR
ts_l, st_l, u_l = simulate_cartpole(state0.copy(), lambda t,s: float(-K_lqr@s), T, dt)

# MPC
Ad, Bd = discretize_linear(A, B, dt)
Qf = 10*Q; R_mpc = np.array([[0.1]])
n_steps = int(T/dt)
st_m = np.zeros((n_steps+1,4)); u_m = np.zeros(n_steps+1)
st_m[0] = state0.copy(); state = state0.copy()
for i in range(n_steps):
    U_opt, _ = solve_mpc(state, Ad, Bd, 20, Q, R_mpc, Qf, -10, 10)
    u_m[i] = U_opt[0,0]
    state = state + dt * cartpole_dynamics(state, u_m[i])
    st_m[i+1] = state
u_m[-1] = u_m[-2]
ts_m = np.linspace(0, T, n_steps+1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, ts, st, u) in zip(axes, [
    ('PID', ts_p, st_p, u_p), ('LQR', ts_l, st_l, u_l), ('MPC', ts_m, st_m, u_m)]):
    ax.plot(ts, st[:, 0], label='x')
    ax.plot(ts, st[:, 1], label='θ')
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.set_title(label); ax.set_xlabel('t [s]')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('State')
fig.suptitle(f'Balancing comparison (θ₀={np.degrees(theta0):.0f}°)', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
animate_cartpole(ts_l, st_l, u_l)

## Swing-Up: Energy-Based vs MPC in Phase Space <a id="swingup-comparison"></a>

In [ ]:
from lib.cartpole_sim import cartpole_energy, upright_energy

# Energy-based swing-up
E_target = upright_energy()
K_sw, _ = solve_lqr(A, B, np.diag([1,10,1,10]), np.array([[1.0]]))

def energy_ctrl(t, state):
    theta, theta_dot = state[1], state[3]
    x, x_dot = state[0], state[2]
    if abs(theta) < 0.3 and abs(theta_dot) < 2.0:
        return float(-K_sw @ state)
    E = cartpole_energy(state)
    u = 20.0 * (E - E_target) * theta_dot * np.cos(theta) - 1.0*x - 1.0*x_dot
    return np.clip(u, -10, 10)

state0_swing = np.array([0.0, np.pi - 0.05, 0.0, 0.0])
ts_e, st_e, u_e = simulate_cartpole(state0_swing, energy_ctrl, T=10.0)

# MPC swing-up (via nonlinear sim + linearised MPC)
n_s = int(10.0/dt); st_ms = np.zeros((n_s+1,4)); u_ms = np.zeros(n_s+1)
st_ms[0] = state0_swing.copy(); state = state0_swing.copy()
for i in range(n_s):
    U_opt, _ = solve_mpc(state, Ad, Bd, 20, Q, R_mpc, Qf, -10, 10)
    u_ms[i] = U_opt[0,0]
    state = state + dt * cartpole_dynamics(state, u_ms[i])
    st_ms[i+1] = state
u_ms[-1] = u_ms[-2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(st_e[:, 1], st_e[:, 3], 'b', lw=0.8, label='Energy-based')
ax1.plot(st_e[0,1], st_e[0,3], 'go', ms=8, label='start (π)')
ax1.plot(0, 0, 'r*', ms=12, label='target (0)')
ax1.set_xlabel('θ'); ax1.set_ylabel('θ̇')
ax1.set_title('Energy-Based Swing-Up')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(st_ms[:, 1], st_ms[:, 3], 'r', lw=0.8, label='Linear MPC')
ax2.plot(st_ms[0,1], st_ms[0,3], 'go', ms=8, label='start (π)')
ax2.plot(0, 0, 'r*', ms=12, label='target (0)')
ax2.set_xlabel('θ'); ax2.set_ylabel('θ̇')
ax2.set_title('MPC Swing-Up (linearised — struggles!)')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

fig.suptitle('Swing-Up Phase Portraits', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
animate_cartpole(ts_e, st_e, u_e)

## Reflection Questions <a id="reflection"></a>

1. **Why does LQR not work for swing-up?** What assumption does it rely on?
2. **MPC with a linear model struggles with swing-up.** What would you need to change to make MPC work for swing-up?
3. **In the control hierarchy table, where would you place an RL policy trained for high-level navigation?**
4. **What is one trade-off between MPC and PID on embedded hardware?**
5. **The Bellman principle appears in LQR (continuous HJB), MPC (truncated DP), and RL (discrete Bellman). What is the key difference between these three uses?**

---

## Final Comparison <a id="final-comparison"></a>

Let's put all balancing controllers side by side on the same initial condition:

In [ ]:
init_compare = np.array([0.0, 0.2, 0.0, 0.0])
controllers = {
    'PID': PIDController(Kp=40, Ki=0, Kd=15, dt=env.dt, x_setpoint=TARGET_X, Kp_x=1.0),
    'LQR': LQRController(env_or_AB=env, target_state=env.target_state),
    'MPC': MPCController(env=env, target_state=env.target_state),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, ctrl in controllers.items():
    ctrl.reset() if hasattr(ctrl, 'reset') else None
    ts_c, st_c, u_c, _ = env.run_episode(ctrl, T=5.0, initial_state=init_compare)
    axes[0].plot(ts_c, st_c[:, 1], label=name)
    axes[1].plot(ts_c, u_c, label=name)

axes[0].set_xlabel('t [s]'); axes[0].set_ylabel('θ [rad]')
axes[0].set_title('Angle'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('t [s]'); axes[1].set_ylabel('u [N]')
axes[1].set_title('Control'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
fig.suptitle('PID vs LQR vs MPC (θ₀ = 0.2 rad)', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
lqr_compare = LQRController(env_or_AB=env, target_state=env.target_state)
ts_lc, st_lc, u_lc, _ = env.run_episode(lqr_compare, T=5.0, initial_state=init_compare)
animate_episode(ts_lc, st_lc, u_lc, env)